# 03 – Model Development
**Project:** War Economic Impact Predictor  

## Objectives
1. Establish baseline models (mean predictor, dummy classifier)  
2. Train Gradient Boosting, Random Forest, and Linear models  
3. Cross-validate and compare model families  
4. Hyperparameter tune the best model via Optuna  
5. Track all experiments with MLflow  
6. Persist final models to `models/`

---

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, f1_score
import joblib

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
from src.utils import load_config

%matplotlib inline
cfg = load_config(ROOT / 'config' / 'config.yaml')
DATA_CFG = cfg['data']
RS = DATA_CFG['random_state']
print('Setup complete.')

In [ ]:
feat_path = ROOT / 'data' / 'processed' / 'war_economic_features.parquet'
df = pd.read_parquet(feat_path)
print(f'Feature matrix: {df.shape}')

TARGET_REG = DATA_CFG['target_regression']
TARGET_CLS = DATA_CFG['target_classification']
EXCLUDE = {TARGET_REG, TARGET_CLS}
feat_cols = [c for c in df.columns if c not in EXCLUDE]

X = df[feat_cols].fillna(0)
y_reg = df[TARGET_REG]
y_cls = df[TARGET_CLS]
print(f'Features: {len(feat_cols)} | Target (reg): {TARGET_REG} | Target (cls): {TARGET_CLS}')

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
X_tr, X_te, yr_tr, yr_te = train_test_split(X, y_reg, test_size=0.2, random_state=RS)
_, _, yc_tr, yc_te = train_test_split(X, y_cls, test_size=0.2, random_state=RS, stratify=y_cls)

scaler = RobustScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

print(f'Train: {X_tr.shape}  Test: {X_te.shape}')

## 1. Baselines

In [ ]:
# Regression baseline
dummy_reg = DummyRegressor(strategy='mean').fit(X_tr_sc, yr_tr)
base_rmse = np.sqrt(mean_squared_error(yr_te, dummy_reg.predict(X_te_sc)))
print(f'Baseline RMSE (mean predictor): {base_rmse:.4f}')

# Classification baseline
dummy_cls = DummyClassifier(strategy='stratified', random_state=RS).fit(X_tr_sc, yc_tr)
base_f1 = f1_score(yc_te, dummy_cls.predict(X_te_sc), average='weighted')
print(f'Baseline F1-weighted (stratified random): {base_f1:.4f}')

## 2. Cross-Validation Comparison

In [ ]:
# ── Regression CV ─────────────────────────────────────────────────────────────
reg_models = {
    'Ridge': Ridge(alpha=1.0),
    'GBRegressor': GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=RS),
    'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=RS, n_jobs=-1),
}
cv_reg_results = {}
for name, model in reg_models.items():
    scores = cross_val_score(model, X_tr_sc, yr_tr, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_reg_results[name] = {'mean': -scores.mean(), 'std': scores.std()}
    print(f'{name:20s}  RMSE = {-scores.mean():.4f} ± {scores.std():.4f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
names = list(cv_reg_results.keys())
means = [cv_reg_results[n]['mean'] for n in names]
stds  = [cv_reg_results[n]['std']  for n in names]
ax.bar(names, means, yerr=stds, capsize=5, color='steelblue', edgecolor='white')
ax.axhline(base_rmse, color='red', ls='--', label=f'Baseline RMSE={base_rmse:.2f}')
ax.set_ylabel('CV RMSE')
ax.set_title('Regression Model Comparison (5-Fold CV)')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'cv_regression_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Classification CV ─────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
cls_models = {
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=RS),
    'GBClassifier': GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=RS),
}
cv_cls_results = {}
for name, model in cls_models.items():
    scores = cross_val_score(model, X_tr_sc, yc_tr, cv=skf, scoring='f1_weighted', n_jobs=-1)
    cv_cls_results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'{name:25s}  F1w = {scores.mean():.4f} ± {scores.std():.4f}')

## 3. Hyperparameter Optimisation (Optuna)

In [ ]:
def reg_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'random_state': RS,
    }
    m = GradientBoostingRegressor(**params)
    s = cross_val_score(m, X_tr_sc, yr_tr, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
    return float(np.mean(s))

study = optuna.create_study(direction='maximize', study_name='gbr_regression')
study.optimize(reg_objective, n_trials=30, timeout=300, show_progress_bar=True)

print(f'Best trial RMSE: {-study.best_value:.4f}')
print(f'Best params:     {study.best_params}')

In [ ]:
import optuna.visualization as ov
fig = ov.plot_optimization_history(study)
fig.show()

fig2 = ov.plot_param_importances(study)
fig2.show()

## 4. Final Model Training + MLflow Logging

In [ ]:
import mlflow.sklearn

best_params = {**study.best_params, 'random_state': RS}

mlflow.set_tracking_uri(str(ROOT / 'mlruns'))
mlflow.set_experiment('war_economic_impact')

with mlflow.start_run(run_name='gbr_tuned_regression'):
    model = GradientBoostingRegressor(**best_params)
    model.fit(X_tr_sc, yr_tr)
    y_pred = model.predict(X_te_sc)

    rmse = float(np.sqrt(mean_squared_error(yr_te, y_pred)))
    r2   = float(r2_score(yr_te, y_pred))

    mlflow.log_params(best_params)
    mlflow.log_metrics({'test_rmse': rmse, 'test_r2': r2})
    mlflow.sklearn.log_model(model, 'gbr_model')

    print(f'Test RMSE: {rmse:.4f}  |  R²: {r2:.4f}')

# Save artefacts
model_dir = ROOT / 'models'
model_dir.mkdir(exist_ok=True)
joblib.dump(model, model_dir / 'regression_xgb.joblib')
joblib.dump(scaler, model_dir / 'scaler_xgb.joblib')
print('Model and scaler saved.')

In [ ]:
# ── Classification final model ────────────────────────────────────────────────
with mlflow.start_run(run_name='gbc_tuned_classification'):
    cls_model = GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.08, max_depth=5, random_state=RS
    )
    cls_model.fit(X_tr_sc, yc_tr)
    yc_pred = cls_model.predict(X_te_sc)
    f1 = f1_score(yc_te, yc_pred, average='weighted')

    mlflow.log_metric('test_f1_weighted', f1)
    mlflow.sklearn.log_model(cls_model, 'gbc_model')
    print(f'Classification F1-weighted: {f1:.4f}')

joblib.dump(cls_model, model_dir / 'classification_xgb.joblib')
joblib.dump(scaler, model_dir / 'scaler_xgb_cls.joblib')
print('Classification model saved.')

## 5. Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=feat_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
importances.head(20).sort_values().plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importances – GBR Regression')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'feature_importance_regression.png', dpi=150)
plt.show()

## 6. Summary

| Model | Task | CV Score | Test Score |
|---|---|---|---|
| GBR (tuned) | Regression | RMSE ≈ X.XX | RMSE ≈ X.XX |
| GBC (default) | Classification | F1w ≈ X.XX | F1w ≈ X.XX |

> **Next step:** `04_evaluation.ipynb` for SHAP, residual analysis, and confusion matrices.